# 예제 franka_ex08: FR3 경로 제약조건 — Orientation Path Constraint

Franka FR3 끝단의 **방향(아래)** 을 *경로 내내* 유지하면서 여러 목표 지점을 순회.
"물컵 운반" 시나리오 — 가는 도중에도 손목이 뒤집히면 안 된다.

## 이 노트북이 강조하는 것 — `path_constraints` 의 도입

ex03~07 의 모든 plan 은 `goal_constraints` 만 채웠다 (시작 / 끝 조건). ex08 은 처음으로
`MotionPlanRequest.path_constraints` 에도 제약을 넣는다 — **경로 도중에도 만족해야 할
조건** 이다.

| Constraints | 언제 만족 | 본 예제에서 |
|---|---|---|
| `goal_constraints` | 도착점에서만 | 위치 + 방향 |
| `path_constraints` | 경로 내내 | **끝단 방향 (아래)** 유지 |

`OrientationConstraint` 의 `absolute_x/y/z_axis_tolerance` 는 라디안 단위 회전 허용량.
yaw 자유도를 풀고 싶으면 `tol_z = π` 정도를 주면 된다.

운영 노하우 — **제약 하 플래닝은 일반 plan 보다 어려우므로**
`num_planning_attempts` / `allowed_planning_time` 을 늘려야 (예: 5→15, 10s→30s).

## 차이를 눈으로 보게 만드는 장치 4가지

`path_constraints` 가 묶는 것은 **방향**인데 EE 경로선은 **위치**만 그린다. 그래서 선 모양만
봐서는 제약 유무가 잘 구분되지 않는다. 이 노트북은 방향을 직접 보여주는 장치를 얹는다.

| 장치 | 어떻게 보이나 |
|---|---|
| **기울기 색칠 경로** | 아래 방향에서 벗어난 각도로 선을 초록→빨강 그라데이션 |
| **방향 화살표** | 경로 위 일정 간격마다 "컵이 생각하는 위쪽"(TCP −z) 화살표 |
| **물컵 마커** | `frame_locked` 로 TCP 에 붙어 손목을 따라 기운다 |
| **기울기 로그 / 그래프** | 최대·평균 기울기를 숫자와 곡선으로 |

제약 경로(`con_A`)와 무제약 경로(`free_A`)는 **마커 namespace 가 달라 화면에 동시에 남는다.**

## 이전 예제와의 관계

ex07 의 `plan_viz_execute()` 패턴 (plan-only → FK 미리보기 → execute) 을 그대로 재사용하고,
**`path_quat` 옵셔널 인자 한 개** 로 path constraint 를 켜고 끈다 — 같은 함수로 제약 하 / 없음
비교가 가능.

## 노트북 구성
1. **로봇 상수**
2. **핵심 — `plan_viz_execute(..., path_quat=...)` 워크플로** ← 이 노트북의 본질
3. **핵심을 쓰기 위한 설정** — ROS init, 노드, 클라이언트(FK 포함), SRDF, Pose / MoveGroup / 마커 헬퍼
4. **시나리오** — 시작점 → A → B → C → 비교(제약 없이 A) → 기울기 그래프 → ready

## 실행 절차

이 노트북은 별도로 띄운 MoveIt + RViz 의 `move_group` 액션 서버에 클라이언트로 붙는다.

> ⚠ 다른 로봇용 MoveIt launch 가 떠 있으면 같은 토픽으로 충돌할 수 있다.
> 시작 전에 `pgrep -af 'ros2 launch'` 로 잔존 프로세스가 없는지 확인하자.

### 터미널 1 — Franka FR3 (Gazebo Sim) + MoveIt + RViz 기동

```bash
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
ros2 launch franka_tutorials franka_gazebo_moveit.launch.py
```

RViz 가 뜨면 **`MarkerArray` Display 를 추가하고 Topic 을 `/constraint_markers` 로 설정**한다.
Fixed Frame 은 `fr3_link0` 로 둔다. 경로선·화살표·물컵이 모두 이 토픽 하나로 온다.

### 터미널 2 — Jupyter 기동

```bash
source ~/venv/ros_jazzy/bin/activate
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
cd ~/robot_arm/src/robotarm_tutorials/robot_arm_tutorials/robot_arm_tutorials
jupyter lab franka_ex08_constraints.ipynb
```

셀을 위에서 아래로 순서대로 실행한다 (`Shift+Enter`).

### 관찰 포인트

- **4-3 (A, 제약 O)** — 경로선이 통째로 초록, 화살표가 전부 위를 향함, 물컵이 수평 유지
- **4-6 (A, 제약 X)** — 같은 시작·목표인데 경로 중간이 붉게 물들고 화살표가 눕는다
- 플래너가 샘플링 기반이라 무제약 결과는 실행마다 다르다. 기울기가 작게 나오면 셀을 다시 실행

## 1. 로봇 상수

In [1]:
PLANNING_GROUP    = 'fr3_arm'
REFERENCE_FRAME   = 'fr3_link0'
END_EFFECTOR_LINK = 'fr3_hand_tcp'
ARM_JOINTS        = ['fr3_joint1', 'fr3_joint2', 'fr3_joint3',
                     'fr3_joint4', 'fr3_joint5', 'fr3_joint6',
                     'fr3_joint7']
MARKER_TOPIC      = '/constraint_markers'

## 2. 핵심 — `path_constraints` 를 얹은 `plan_viz_execute`

이 노트북이 가장 먼저 정의해야 하는 함수들. 다섯이 한 묶음으로 본 예제의 본질을 이룬다.

- `make_orientation_path_constraint()` — 경로 내내 유지할 방향 제약 빌더
- `plan_to_pose_goal()` — `path_quat` 옵셔널 인자로 path_constraints 토글
- `trajectory_to_ee_path()` / `publish_ee_path()` / `execute_trajectory()` — ex07 과 동일
- `plan_viz_execute()` — 위 모두를 한 줄로 묶고 `path_quat` 유무에 따라 EE 경로 색을 바꿈

함수 *정의* 시점엔 setup 객체 (`node`, `move_client` 등) 를 lookup 하지 않으므로 객체가
아직 없어도 OK — 호출은 4 절(시나리오) 에서 일어난다.

### 2-1. 핵심에 필요한 import

In [ ]:
import math
import numpy as np
import rclpy
import tf_transformations
from moveit_msgs.action import MoveGroup, ExecuteTrajectory
from moveit_msgs.srv import GetPositionFK
from moveit_msgs.msg import (
    Constraints, MoveItErrorCodes,
    MotionPlanRequest, PlanningOptions, RobotState,
    OrientationConstraint,
)
from visualization_msgs.msg import Marker
from geometry_msgs.msg import Point, Quaternion, Vector3
from std_msgs.msg import ColorRGBA

### 2-2. `make_orientation_path_constraint()` — 경로 방향 제약 빌더

`goal_constraints` 의 OrientationConstraint 와 자료형은 같지만 의미가 다르다 —
**경로 내내** 이 방향을 (지정 tolerance 안에서) 유지해야 한다.

`tol_xy` 는 X / Y 축 (roll / pitch), `tol_z` 는 Z 축 (yaw) 허용 회전량 (rad).
yaw 자유도를 풀고 싶으면 `tol_z = math.pi` — 손목이 어떤 방향이든 허용.

In [3]:
def make_orientation_path_constraint(quat: Quaternion,
                                     tol_xy: float,
                                     tol_z: float) -> OrientationConstraint:
    oc = OrientationConstraint()
    oc.header.frame_id = REFERENCE_FRAME
    oc.link_name = END_EFFECTOR_LINK
    oc.orientation = quat
    oc.absolute_x_axis_tolerance = tol_xy
    oc.absolute_y_axis_tolerance = tol_xy
    oc.absolute_z_axis_tolerance = tol_z
    oc.weight = 1.0
    return oc

### 2-3. `send_move_goal()` + `plan_to_pose_goal()` — `path_quat` 옵셔널

> **목표 자세 허용치 주의** — `goal_constraints` 의 yaw 허용치를 `path_constraints` 와
> 같은 `tol_z` 로 맞춘다. 경로는 yaw 자유인데 목표만 ±0.57° 로 묶으면, 플래너가 경로 내내
> 자유롭게 굴린 yaw 를 도착 직전에 되감아야 해서 **경로가 크게 우회**한다.
> 또한 제약 유무와 상관없이 goal 이 동일해야 4-3 과 4-6 의 비교가 공정하다.

ex07 의 plan-only wrapper 에 `path_quat` 인자 하나를 추가했다.
이 인자가 주어지면 `req.path_constraints` 에 OrientationConstraint 가 들어가고,
없으면 ex07 과 동일하게 동작한다.

**제약 하 플래닝은 비싸므로** 호출 측에서 `attempts` / `plan_time` 을 늘려 부르는 게 좋다
(시나리오에서는 15회 / 30초).

In [ ]:
def send_move_goal(req: MotionPlanRequest, plan_only: bool = False):
    goal = MoveGroup.Goal()
    goal.request = req
    goal.planning_options = PlanningOptions(
        plan_only=plan_only,
        replan=not plan_only,
        replan_attempts=3 if not plan_only else 0,
    )
    sf = move_client.send_goal_async(goal)
    rclpy.spin_until_future_complete(node, sf)
    handle = sf.result()
    if handle is None or not handle.accepted:
        return MoveItErrorCodes.PLANNING_FAILED, None
    rf = handle.get_result_async()
    rclpy.spin_until_future_complete(node, rf)
    res = rf.result().result
    return res.error_code.val, res.planned_trajectory


def plan_to_joint_goal(joint_values: dict, vel: float = 0.3, acc: float = 0.3,
                       plan_time: float = 10.0, attempts: int = 5):
    req = make_plan_request(vel, acc, attempts=attempts, plan_time=plan_time)
    req.goal_constraints.append(make_joint_constraints(joint_values))
    code_val, traj = send_move_goal(req, plan_only=True)
    return code_val == MoveItErrorCodes.SUCCESS, traj


def plan_to_pose_goal(pose, vel: float = 0.3, acc: float = 0.3,
                      plan_time: float = 10.0, attempts: int = 5,
                      path_quat=None, tol_xy: float = 0.3, tol_z: float = 3.14):
    '''path_quat 가 주어지면 경로 내내 그 방향이 유지되도록 path_constraints 추가.'''
    req = make_plan_request(vel, acc, attempts=attempts, plan_time=plan_time)
    if path_quat is not None:
        path_c = Constraints()
        path_c.orientation_constraints.append(
            make_orientation_path_constraint(path_quat, tol_xy, tol_z)
        )
        req.path_constraints = path_c
    goal_c = Constraints()
    goal_c.position_constraints.append(make_position_constraint(pose))
    # 목표 yaw 허용치를 경로 제약과 같게 맞춘다.
    #   - path 는 yaw 자유(tol_z=π)인데 goal 만 ±0.57° 로 못박으면 마지막에 크게 되감김
    #   - 제약 유무와 무관하게 같은 goal 을 써야 4-3 vs 4-6 비교가 공정하다
    goal_c.orientation_constraints.append(
        make_orientation_constraint(pose, tol_z=tol_z)
    )
    req.goal_constraints.append(goal_c)
    code_val, traj = send_move_goal(req, plan_only=True)
    return code_val == MoveItErrorCodes.SUCCESS, traj

### 2-4. `trajectory_to_ee_path()` — FK 로 EE **위치 + 방향** 환산

ex07 은 위치만 모았지만, 여기서는 `pose_stamped[0].pose.orientation` 도 함께 담는다.
**경로 제약이 묶는 대상이 방향이므로, 방향을 버리면 보여줄 것이 없다.**
반환값은 `[((x, y, z), (qx, qy, qz, qw)), ...]`.

In [ ]:
def trajectory_to_ee_path(trajectory, max_points: int = 60):
    '''RobotTrajectory → [((x,y,z), (qx,qy,qz,qw)), ...] (fr3_hand_tcp, base 기준).'''
    jt = trajectory.joint_trajectory
    total = len(jt.points)
    if total == 0:
        return []
    step = max(1, total // max_points)
    indices = list(range(0, total, step))
    if indices[-1] != total - 1:
        indices.append(total - 1)
    pts = []
    for idx in indices:
        req = GetPositionFK.Request()
        req.header.frame_id = REFERENCE_FRAME
        req.fk_link_names = [END_EFFECTOR_LINK]
        rs = RobotState()
        rs.joint_state.name = list(jt.joint_names)
        rs.joint_state.position = list(jt.points[idx].positions)
        req.robot_state = rs
        fut = fk_client.call_async(req)
        rclpy.spin_until_future_complete(node, fut)
        resp = fut.result()
        if resp and resp.error_code.val == MoveItErrorCodes.SUCCESS and resp.pose_stamped:
            ps = resp.pose_stamped[0].pose
            p, q = ps.position, ps.orientation
            pts.append(((p.x, p.y, p.z), (q.x, q.y, q.z, q.w)))   # 방향까지 보관
    return pts

### 2-4b. 기울기 각도 — 방향 제약을 숫자로

"아래를 향한다" 는 **TCP 의 z 축이 월드 −Z 와 이루는 각** 으로 잰다. 0° 면 완벽히 수직,
90° 면 손목이 옆으로 누운 것.

```
z_axis(월드) = R(q) · [0, 0, 1]
기울기       = arccos( −z_axis[2] )
```

이 각도 하나로 세 가지를 얻는다 — 경로선 색(초록↔빨강), 화살표 색, 그리고 로그·그래프용 수치.
`TILT_LOG` 에 namespace 별로 쌓아두면 마지막에 두 곡선을 겹쳐 그릴 수 있다.

In [ ]:
TILT_LOG = {}          # ns -> 기울기 각도 리스트 (마지막 그래프 셀에서 사용)


def tilt_deg(quat) -> float:
    '''TCP z축이 "아래(-Z)" 에서 몇 도 벗어났나. 0° = 완벽히 아래.'''
    z_axis = tf_transformations.quaternion_matrix(quat)[:3, 2]
    return math.degrees(math.acos(float(np.clip(-z_axis[2], -1.0, 1.0))))


def tcp_up(quat):
    '''TCP 가 생각하는 "위쪽"(-z) 을 월드 좌표 단위벡터로.'''
    return -tf_transformations.quaternion_matrix(quat)[:3, 2]


def tilt_color(deg: float, span: float = 60.0, alpha: float = 0.95) -> ColorRGBA:
    '''0° 초록 → span° 이상 빨강.'''
    t = min(max(deg / span, 0.0), 1.0)
    return ColorRGBA(r=float(t), g=float(1.0 - t), b=0.15, a=alpha)


def report_tilt(ee_path, ns: str, label: str, constrained: bool,
                tol_xy: float = None):
    tilts = [tilt_deg(q) for _, q in ee_path]
    TILT_LOG[ns] = tilts
    tag = '제약 O' if constrained else '제약 X'
    msg = (f'[{label} · {tag}] 기울기 최대 {max(tilts):5.1f}° / '
           f'평균 {sum(tilts) / len(tilts):5.1f}°  (샘플 {len(tilts)}개)')
    if constrained and tol_xy is not None:
        msg += f'  — 허용 ±{math.degrees(tol_xy):.0f}°'
    node.get_logger().info(msg)
    return tilts

### 2-5. `publish_ee_path()` + `publish_ee_frames()` + `execute_trajectory()`

- `publish_ee_path` — **기울기로 점마다 색을 칠한** LINE_STRIP. `Marker.colors` 는
  `points` 와 같은 길이면 점별 색으로 그려진다.
- `publish_ee_frames` — 경로 위 일정 간격마다 TCP "위쪽" 화살표. 제약이 걸리면 전부
  나란히 서고, 없으면 중간에서 눕는다.
- 두 함수 모두 `ns` 를 인자로 받는다 → **제약 경로와 무제약 경로가 서로를 지우지 않는다.**

In [ ]:
def publish_ee_path(ee_path, ns: str = 'ee_path', color=None,
                    tilt_colored: bool = True, width: float = 0.012):
    '''EE 경로를 LINE_STRIP 으로. tilt_colored 면 점마다 기울기 색.'''
    if not ee_path:
        return
    line = Marker()
    line.header.frame_id = REFERENCE_FRAME
    line.header.stamp = node.get_clock().now().to_msg()
    line.ns = ns
    line.id = 0
    line.type = Marker.LINE_STRIP
    line.action = Marker.ADD
    line.pose.orientation.w = 1.0
    line.scale.x = width
    line.points = [Point(x=p[0], y=p[1], z=p[2]) for p, _ in ee_path]
    if tilt_colored:
        line.color = ColorRGBA(r=1.0, g=1.0, b=1.0, a=1.0)      # colors 가 우선
        line.colors = [tilt_color(tilt_deg(q)) for _, q in ee_path]
    else:
        line.color = color if color is not None else COLOR_EE_CONSTRAINED
    _markers.markers = [m for m in _markers.markers if (m.ns, m.id) != (ns, 0)]
    _markers.markers.append(line)
    marker_pub.publish(_markers)


def publish_ee_frames(ee_path, ns: str = 'ee_frames',
                      every: int = 6, length: float = 0.08):
    '''경로 위 일정 간격마다 TCP "위쪽"(-z) 화살표를 세운다.'''
    _markers.markers = [m for m in _markers.markers if m.ns != ns]
    if not ee_path:
        marker_pub.publish(_markers)
        return
    stamp = node.get_clock().now().to_msg()
    idxs = list(range(0, len(ee_path), max(1, every)))
    if idxs[-1] != len(ee_path) - 1:
        idxs.append(len(ee_path) - 1)
    for i in idxs:
        (x, y, z), q = ee_path[i]
        up = tcp_up(q)
        m = Marker()
        m.header.frame_id = REFERENCE_FRAME
        m.header.stamp = stamp
        m.ns = ns
        m.id = i
        m.type = Marker.ARROW
        m.action = Marker.ADD
        m.pose.orientation.w = 1.0
        m.points = [
            Point(x=x, y=y, z=z),
            Point(x=x + up[0] * length, y=y + up[1] * length, z=z + up[2] * length),
        ]
        m.scale = Vector3(x=0.008, y=0.018, z=0.02)   # 자루 지름, 머리 지름, 머리 길이
        m.color = tilt_color(tilt_deg(q))
        _markers.markers.append(m)
    marker_pub.publish(_markers)


def clear_path_markers():
    '''경로 / 화살표 마커만 모두 지운다 (타깃·물컵은 유지).'''
    _markers.markers = [m for m in _markers.markers
                        if not (m.ns.startswith('con_') or m.ns.startswith('free_'))]
    marker_pub.publish(_markers)


def execute_trajectory(trajectory) -> bool:
    g = ExecuteTrajectory.Goal()
    g.trajectory = trajectory
    sf = execute_client.send_goal_async(g)
    rclpy.spin_until_future_complete(node, sf)
    handle = sf.result()
    if handle is None or not handle.accepted:
        return False
    rf = handle.get_result_async()
    rclpy.spin_until_future_complete(node, rf)
    return rf.result().result.error_code.val == MoveItErrorCodes.SUCCESS

### 2-6. `plan_viz_execute()` — 계획 → 기울기 측정 → 시각화 → 실행

ex07 과 흐름은 같고 **가운데에 "기울기 측정 + 색칠/화살표"** 가 들어갔다.

- `path_quat` 유무로 `ns` 가 `con_<label>` / `free_<label>` 로 갈린다 → 두 경로가 공존
- `show=False` 로 부르면 마커를 남기지 않는다 (시작점 복귀 이동처럼 비교와 무관한 구간)

In [ ]:
def plan_viz_execute(pose, vel: float = 0.3, label: str = '',
                     path_quat=None, tol_xy: float = 0.3, tol_z: float = 3.14,
                     attempts: int = 5, plan_time: float = 10.0,
                     show: bool = True) -> bool:
    constrained = path_quat is not None
    ok, traj = plan_to_pose_goal(
        pose, vel=vel, acc=vel,
        plan_time=plan_time, attempts=attempts,
        path_quat=path_quat, tol_xy=tol_xy, tol_z=tol_z,
    )
    if not ok or traj is None:
        node.get_logger().error(
            f'{label}: 계획 실패 (제약: {"있음" if constrained else "없음"})'
        )
        return False

    ee = trajectory_to_ee_path(traj)
    if ee:
        ns = f'{"con" if constrained else "free"}_{label}'
        report_tilt(ee, ns, label, constrained, tol_xy)   # 기울기는 항상 측정·기록
        if show:                                          # 마커만 선택적으로
            publish_ee_path(ee, ns=ns)
            publish_ee_frames(ee, ns=f'{ns}_ax', every=6)
    return execute_trajectory(traj)


def plan_viz_execute_joint(joint_values: dict, vel: float = 0.3,
                           label: str = '', show: bool = False) -> bool:
    ok, traj = plan_to_joint_goal(joint_values, vel=vel, acc=vel)
    if not ok or traj is None:
        node.get_logger().error(f'{label}: 계획 실패')
        return False
    ee = trajectory_to_ee_path(traj)
    if ee and show:
        publish_ee_path(ee, ns=f'free_{label}')
    return execute_trajectory(traj)

## 3. 핵심을 쓰기 위한 설정

위 핵심 함수들이 참조하는 객체와 보조 헬퍼.

- ROS 2 초기화 / 노드 / 액션·서비스 클라이언트 / `joint_states` 구독 / 마커 퍼블리셔 / FK 클라이언트
- 서버 / `joint_states` 준비 대기
- SRDF `ready` 자세
- Pose 헬퍼
- MoveGroup 빌딩블록 (Constraints / `MotionPlanRequest`)
- 마커 — 시작점 / 타깃 / 허용 오차 링 + EE 경로 색상 상수

### 3-1. ROS 2 초기화 + 노드 + 클라이언트

In [8]:
from rclpy.node import Node
from rclpy.action import ActionClient
from rclpy.parameter import Parameter
from sensor_msgs.msg import JointState
from visualization_msgs.msg import MarkerArray

try:
    rclpy.init()
except RuntimeError:
    pass

node = Node(
    'franka_ex08_constraints_demo',
    parameter_overrides=[Parameter('use_sim_time', value=True)],
)
move_client    = ActionClient(node, MoveGroup, 'move_action')
execute_client = ActionClient(node, ExecuteTrajectory, 'execute_trajectory')
fk_client      = node.create_client(GetPositionFK, 'compute_fk')
marker_pub     = node.create_publisher(MarkerArray, MARKER_TOPIC, 10)

joint_state = {'msg': None}
node.create_subscription(
    JointState, 'joint_states',
    lambda msg: joint_state.update(msg=msg), 10,
)
node.get_logger().info('=== franka_ex08 노트북 노드 생성 완료 ===')

[INFO] [1785210675.243604873] [franka_ex08_constraints_demo]: === franka_ex08 노트북 노드 생성 완료 ===


True

### 3-2. 액션 / 서비스 / `/joint_states` 준비 대기

In [9]:
import time

def wait_for_ready(timeout_sec: float = 30.0) -> None:
    if not move_client.wait_for_server(timeout_sec=timeout_sec):
        raise RuntimeError('MoveGroup 액션 서버 연결 실패')
    if not execute_client.wait_for_server(timeout_sec=timeout_sec):
        raise RuntimeError('ExecuteTrajectory 액션 서버 연결 실패')
    if not fk_client.wait_for_service(timeout_sec=timeout_sec):
        raise RuntimeError('compute_fk 서비스 연결 실패')
    start = time.time()
    while joint_state['msg'] is None:
        rclpy.spin_once(node, timeout_sec=0.1)
        if time.time() - start > timeout_sec:
            raise RuntimeError('joint_states 수신 실패')
    node.get_logger().info(
        'move/execute action + compute_fk svc + /joint_states 준비됨'
    )

wait_for_ready()

[INFO] [1785210677.028818168] [franka_ex08_constraints_demo]: move/execute action + compute_fk svc + /joint_states 준비됨


### 3-3. SRDF 에서 `ready` 자세 읽어오기

In [10]:
from rclpy.parameter_client import AsyncParameterClient
import xml.etree.ElementTree as ET

def fetch_srdf_xml(timeout_sec: float = 10.0) -> str:
    client = AsyncParameterClient(node, 'move_group')
    if not client.wait_for_services(timeout_sec=timeout_sec):
        raise RuntimeError('move_group 파라미터 서비스 연결 실패')
    future = client.get_parameters(['robot_description_semantic'])
    rclpy.spin_until_future_complete(node, future, timeout_sec=timeout_sec)
    return future.result().values[0].string_value

def parse_named_pose(srdf_xml: str, name: str, group: str) -> dict:
    root = ET.fromstring(srdf_xml)
    for gs in root.findall('group_state'):
        if gs.attrib.get('group') == group and gs.attrib.get('name') == name:
            return {j.attrib['name']: float(j.attrib.get('value', '0'))
                    for j in gs.findall('joint')}
    raise RuntimeError(f'SRDF group_state "{name}" (group={group}) 없음')

def load_named_pose(name: str, timeout_sec: float = 10.0) -> dict:
    return parse_named_pose(fetch_srdf_xml(timeout_sec), name, PLANNING_GROUP)

ready_target = load_named_pose('ready')
node.get_logger().info(f'ready: {ready_target}')

[INFO] [1785210678.219219361] [franka_ex08_constraints_demo]: ready: {'fr3_joint1': 0.0, 'fr3_joint2': -0.7853981633974483, 'fr3_joint3': 0.0, 'fr3_joint4': -2.356194490192345, 'fr3_joint5': 0.0, 'fr3_joint6': 1.5707963267948966, 'fr3_joint7': 0.7853981633974483}


True

### 3-4. Pose 헬퍼 — Euler ↔ Quaternion

In [11]:
import math
import tf_transformations
from geometry_msgs.msg import Pose

def euler_to_quaternion(roll: float, pitch: float, yaw: float) -> Quaternion:
    q = tf_transformations.quaternion_from_euler(roll, pitch, yaw)
    return Quaternion(x=q[0], y=q[1], z=q[2], w=q[3])

def make_pose(x: float, y: float, z: float,
              roll: float = 0.0, pitch: float = 0.0, yaw: float = 0.0) -> Pose:
    pose = Pose()
    pose.position = Point(x=x, y=y, z=z)
    pose.orientation = euler_to_quaternion(roll, pitch, yaw)
    return pose

### 3-5. MoveGroup 빌딩블록 — Constraints / `MotionPlanRequest`

In [ ]:
from moveit_msgs.msg import (
    JointConstraint,
    PositionConstraint, BoundingVolume,
)
from shape_msgs.msg import SolidPrimitive
from geometry_msgs.msg import Vector3

def make_joint_constraints(joint_values: dict, tol: float = 0.01) -> Constraints:
    c = Constraints()
    for jname, val in joint_values.items():
        c.joint_constraints.append(JointConstraint(
            joint_name=jname, position=val,
            tolerance_above=tol, tolerance_below=tol, weight=1.0,
        ))
    return c

def make_position_constraint(pose: Pose, tol: float = 0.01) -> PositionConstraint:
    pc = PositionConstraint()
    pc.header.frame_id = REFERENCE_FRAME
    pc.link_name = END_EFFECTOR_LINK
    pc.target_point_offset = Vector3(x=0.0, y=0.0, z=0.0)
    bv = BoundingVolume()
    sphere = SolidPrimitive()
    sphere.type = SolidPrimitive.SPHERE
    sphere.dimensions = [tol]
    bv.primitives.append(sphere)
    sp = Pose()
    sp.position = Point(x=pose.position.x, y=pose.position.y, z=pose.position.z)
    sp.orientation.w = 1.0
    bv.primitive_poses.append(sp)
    pc.constraint_region = bv
    pc.weight = 1.0
    return pc

def make_orientation_constraint(pose_or_quat, tol: float = 0.01,
                                tol_z: float = None) -> OrientationConstraint:
    '''goal_constraints 용 — pose 의 orientation 또는 Quaternion 직접 받음.

    tol_z 를 주면 yaw(z축) 만 따로 풀 수 있다. 경로 제약이 yaw 자유인데 목표만
    yaw 를 못박으면, 플래너가 마지막에 joint7 을 크게 되감으며 경로가 우회한다.
    '''
    oc = OrientationConstraint()
    oc.header.frame_id = REFERENCE_FRAME
    oc.link_name = END_EFFECTOR_LINK
    if hasattr(pose_or_quat, 'orientation'):
        oc.orientation = pose_or_quat.orientation
    else:
        oc.orientation = pose_or_quat
    oc.absolute_x_axis_tolerance = tol
    oc.absolute_y_axis_tolerance = tol
    oc.absolute_z_axis_tolerance = tol if tol_z is None else tol_z
    oc.weight = 1.0
    return oc

def make_plan_request(vel: float = 0.3, acc: float = 0.3,
                      attempts: int = 5, plan_time: float = 10.0,
                      planner_id: str = '') -> MotionPlanRequest:
    req = MotionPlanRequest()
    req.group_name = PLANNING_GROUP
    req.num_planning_attempts = attempts
    req.allowed_planning_time = plan_time
    req.max_velocity_scaling_factor = vel
    req.max_acceleration_scaling_factor = acc
    if planner_id:
        req.planner_id = planner_id
    return req

### 3-6. RViz 마커 — 시작점 / 타깃 / 허용 오차 링 + **물컵**

`publish_cup()` 이 핵심 장치다. 마커의 `header.frame_id` 를 `fr3_hand_tcp` 로 두고
**`frame_locked = True`** 를 켜면, RViz 가 매 프레임 TF 로 다시 그려 **컵이 손목을 따라 기운다.**
`frame_locked` 를 빼면 처음 발행된 자리에 그대로 굳어버리니 반드시 켤 것.

컵은 순수 시각화 마커라 planning scene 에는 없다 — 플래너는 컵의 존재를 모른다.

In [ ]:
from std_msgs.msg import ColorRGBA

COLOR_TEXT          = ColorRGBA(r=1.0, g=1.0, b=1.0, a=1.0)
COLOR_START         = ColorRGBA(r=1.0, g=0.8, b=0.0, a=1.0)
COLOR_EE_CONSTRAINED = ColorRGBA(r=0.0, g=0.9, b=0.9, a=0.95)  # 청록
COLOR_EE_FREE        = ColorRGBA(r=0.6, g=0.6, b=0.6, a=0.85)  # 회색
TARGET_COLORS = [
    ColorRGBA(r=0.9, g=0.2, b=0.2, a=1.0),   # A 빨강
    ColorRGBA(r=0.2, g=0.8, b=0.2, a=1.0),   # B 초록
    ColorRGBA(r=0.2, g=0.3, b=0.9, a=1.0),   # C 파랑
]

_markers = MarkerArray()

def publish_targets(start_pos, targets, tol_rad: float):
    '''시작점, 타깃 A/B/C, 허용 오차 링을 발행. ee_path 마커는 건드리지 않음.'''
    stamp = node.get_clock().now().to_msg()
    target_ns = {'pt', 'txt', 'ring'}
    _markers.markers = [m for m in _markers.markers if m.ns not in target_ns]

    sx, sy, sz = start_pos
    s_sphere = Marker()
    s_sphere.header.frame_id = REFERENCE_FRAME
    s_sphere.header.stamp = stamp
    s_sphere.ns = 'pt'
    s_sphere.id = 0
    s_sphere.type = Marker.SPHERE
    s_sphere.action = Marker.ADD
    s_sphere.pose.position = Point(x=sx, y=sy, z=sz)
    s_sphere.pose.orientation.w = 1.0
    s_sphere.scale = Vector3(x=0.04, y=0.04, z=0.04)
    s_sphere.color = COLOR_START
    s_text = Marker()
    s_text.header.frame_id = REFERENCE_FRAME
    s_text.header.stamp = stamp
    s_text.ns = 'txt'
    s_text.id = 0
    s_text.type = Marker.TEXT_VIEW_FACING
    s_text.action = Marker.ADD
    s_text.pose.position = Point(x=sx, y=sy, z=sz + 0.08)
    s_text.pose.orientation.w = 1.0
    s_text.scale.z = 0.04
    s_text.color = COLOR_TEXT
    s_text.text = 'Start'
    _markers.markers.extend([s_sphere, s_text])

    for i, tgt in enumerate(targets):
        tx, ty, tz = tgt['pos']
        c = TARGET_COLORS[i % len(TARGET_COLORS)]
        sphere = Marker()
        sphere.header.frame_id = REFERENCE_FRAME
        sphere.header.stamp = stamp
        sphere.ns = 'pt'
        sphere.id = i + 1
        sphere.type = Marker.SPHERE
        sphere.action = Marker.ADD
        sphere.pose.position = Point(x=tx, y=ty, z=tz)
        sphere.pose.orientation.w = 1.0
        sphere.scale = Vector3(x=0.04, y=0.04, z=0.04)
        sphere.color = c
        text = Marker()
        text.header.frame_id = REFERENCE_FRAME
        text.header.stamp = stamp
        text.ns = 'txt'
        text.id = i + 1
        text.type = Marker.TEXT_VIEW_FACING
        text.action = Marker.ADD
        text.pose.position = Point(x=tx, y=ty, z=tz + 0.08)
        text.pose.orientation.w = 1.0
        text.scale.z = 0.04
        text.color = COLOR_TEXT
        text.text = tgt['label']
        ring = Marker()
        ring.header.frame_id = REFERENCE_FRAME
        ring.header.stamp = stamp
        ring.ns = 'ring'
        ring.id = i
        ring.type = Marker.CYLINDER
        ring.action = Marker.ADD
        ring.pose.position = Point(x=tx, y=ty, z=tz)
        ring.pose.orientation.w = 1.0
        ring.scale.x = tol_rad * 0.4
        ring.scale.y = tol_rad * 0.4
        ring.scale.z = 0.005
        ring.color = ColorRGBA(r=c.r, g=c.g, b=c.b, a=0.25)
        _markers.markers.extend([sphere, text, ring])

    marker_pub.publish(_markers)


def publish_cup(show: bool = True):
    '''그리퍼(TCP)에 물컵을 붙인다. frame_locked=True 라 손목을 따라 기운다.'''
    _markers.markers = [m for m in _markers.markers if m.ns != 'cup']
    if show:
        stamp = node.get_clock().now().to_msg()
        # 그리퍼가 아래를 보므로 TCP +z 가 월드 아래쪽 → 컵은 TCP +z 방향에 매단다
        glass = Marker()
        glass.header.frame_id = END_EFFECTOR_LINK
        glass.header.stamp = stamp
        glass.ns, glass.id = 'cup', 0
        glass.type = Marker.CYLINDER
        glass.action = Marker.ADD
        glass.frame_locked = True                       # ← 이게 없으면 안 따라간다
        glass.pose.position = Point(x=0.0, y=0.0, z=0.075)
        glass.pose.orientation.w = 1.0
        glass.scale = Vector3(x=0.075, y=0.075, z=0.13)
        glass.color = ColorRGBA(r=0.88, g=0.95, b=1.0, a=0.35)

        water = Marker()
        water.header.frame_id = END_EFFECTOR_LINK
        water.header.stamp = stamp
        water.ns, water.id = 'cup', 1
        water.type = Marker.CYLINDER
        water.action = Marker.ADD
        water.frame_locked = True
        water.pose.position = Point(x=0.0, y=0.0, z=0.105)   # 컵의 월드-아래쪽 절반
        water.pose.orientation.w = 1.0
        water.scale = Vector3(x=0.066, y=0.066, z=0.07)
        water.color = ColorRGBA(r=0.15, g=0.55, b=0.95, a=0.9)

        _markers.markers.extend([glass, water])
    marker_pub.publish(_markers)

## 4. 시나리오 — 그리퍼 아래 방향 유지하고 A / B / C 순회

**좌우로 크게 스윙**시키는 것이 핵심이다. 짧은 이동은 무제약 플래너도 손목을 그대로 두기 때문에
차이가 안 난다. 시작점(우측)에서 A(좌측)까지 **y 방향으로 60cm** 를 건너가게 하면, 무제약
플랜은 최단 관절 경로를 택하며 중간에 손목을 눕히는 해를 자주 고른다.

### 4-1. ready 자세 + 시작점 / 타깃 / 제약 정의 + 물컵 부착

In [ ]:
node.get_logger().info('--- ready 자세로 초기 이동 ---')
plan_viz_execute_joint(ready_target, vel=0.3, label='Ready')
time.sleep(1.0)

# 좌우로 크게 스윙 — 무제약 플랜이 손목을 눕히도록 유도한다
start_pos  = (0.45, -0.25, 0.35)
start_pose = make_pose(*start_pos, math.pi, 0.0, 0.0)

down_quaternion = euler_to_quaternion(math.pi, 0.0, 0.0)  # 그리퍼 아래
TOL   = 0.3        # X/Y 회전 허용 ±17°
TOL_Z = math.pi    # yaw 자유

targets = [
    {'label': 'A', 'pos': (0.45,  0.25, 0.35)},   # 반대편까지 50cm 스윙 ← 핵심 비교 구간
    {'label': 'B', 'pos': (0.35,  0.30, 0.55)},   # 좌측 높은 곳
    {'label': 'C', 'pos': (0.55,  0.00, 0.28)},   # 전방 낮은 곳
]

publish_targets(start_pos, targets, TOL)
publish_cup()          # 그리퍼에 물컵 부착 (frame_locked → 손목을 따라 기운다)

node.get_logger().info(
    f'방향 제약: 아래(roll=π), 허용 ±{math.degrees(TOL):.0f}°, yaw 자유'
)
node.get_logger().info(
    f'스윙 폭: y {start_pos[1]:+.2f} → {targets[0]["pos"][1]:+.2f} m'
)

### 4-2. 시작 위치로 (제약 없이) 이동

이 구간은 비교 대상이 아니므로 `show=False` 로 마커를 남기지 않는다.

In [ ]:
node.get_logger().info('--- 시작 위치로 이동 (제약 없이) ---')
plan_viz_execute(start_pose, label='Start', show=False)
time.sleep(1.0)

### 4-3. 타깃 A — 경로 제약 하 이동  ← **비교의 기준**

`clear_path_markers()` 로 화면을 정리하고 이 경로만 남긴다. 4-6 의 무제약 경로와
**나란히 놓고 볼 수 있도록** 여기서부터 지우지 않는다.

물컵을 보라. 좌우 50cm 를 건너가는 내내 **수평을 유지**한다. 경로선은 통째로 초록,
화살표는 전부 위. 로그의 `최대 기울기` 가 허용치(±17°) 안인지 확인.

In [ ]:
clear_path_markers()      # 이전 경로/화살표를 지우고 시작

tgt = targets[0]
node.get_logger().info(f"--- {tgt['label']} 로 제약 이동 ---")
ok = plan_viz_execute(
    make_pose(*tgt['pos'], math.pi, 0.0, 0.0),
    path_quat=down_quaternion, tol_xy=TOL, tol_z=TOL_Z,
    attempts=15, plan_time=30.0, vel=0.2,
    label=tgt['label'],
)
node.get_logger().info(f"  결과: {'성공' if ok else '실패'}")
time.sleep(1.0)

# 복귀 이동은 비교와 무관 → 마커를 남기지 않는다
plan_viz_execute(
    start_pose,
    path_quat=down_quaternion, tol_xy=TOL, tol_z=TOL_Z,
    attempts=15, plan_time=30.0, vel=0.2,
    label='Back', show=False,
)
time.sleep(0.5)

### 4-4. 타깃 B — 경로 제약 하 이동

`show=False` — 화면은 A 비교용으로 비워두고, **물컵이 계속 수평인지**를 눈으로 본다.
기울기 수치는 로그에 그대로 남는다.

In [ ]:
tgt = targets[1]
node.get_logger().info(f"--- {tgt['label']} 로 제약 이동 ---")
ok = plan_viz_execute(
    make_pose(*tgt['pos'], math.pi, 0.0, 0.0),
    path_quat=down_quaternion, tol_xy=TOL, tol_z=TOL_Z,
    attempts=15, plan_time=30.0, vel=0.2,
    label=tgt['label'], show=False,     # 화면은 A 비교용으로 비워둔다
)
node.get_logger().info(f"  결과: {'성공' if ok else '실패'}")
time.sleep(1.0)

# 복귀 이동은 비교와 무관 → 마커를 남기지 않는다
plan_viz_execute(
    start_pose,
    path_quat=down_quaternion, tol_xy=TOL, tol_z=TOL_Z,
    attempts=15, plan_time=30.0, vel=0.2,
    label='Back', show=False,
)
time.sleep(0.5)

### 4-5. 타깃 C — 경로 제약 하 이동

여기도 `show=False`. 낮은 전방 자세에서도 컵이 수평을 유지하는지 확인.

In [ ]:
tgt = targets[2]
node.get_logger().info(f"--- {tgt['label']} 로 제약 이동 ---")
ok = plan_viz_execute(
    make_pose(*tgt['pos'], math.pi, 0.0, 0.0),
    path_quat=down_quaternion, tol_xy=TOL, tol_z=TOL_Z,
    attempts=15, plan_time=30.0, vel=0.2,
    label=tgt['label'], show=False,     # 화면은 A 비교용으로 비워둔다
)
node.get_logger().info(f"  결과: {'성공' if ok else '실패'}")
time.sleep(1.0)

# 복귀 이동은 비교와 무관 → 마커를 남기지 않는다
plan_viz_execute(
    start_pose,
    path_quat=down_quaternion, tol_xy=TOL, tol_z=TOL_Z,
    attempts=15, plan_time=30.0, vel=0.2,
    label='Back', show=False,
)
time.sleep(0.5)

### 4-6. 비교 — **같은 A 로, 제약만 빼고** 이동

`path_quat` 인자를 안 넘기면 ex07 과 동일한 일반 plan 이다. 시작점도 목표도 4-3 과 완전히
같고 **오직 `path_constraints` 유무만 다르다.**

- 4-3 의 초록 경로(`con_A`)는 **그대로 화면에 남아 있다** — namespace 가 다르기 때문
- 이번 경로(`free_A`)는 기울어진 구간이 **붉게** 물들고 화살표가 눕는다
- 실행 중 **물컵이 옆으로 기우는 순간**이 이 예제의 결론이다

> 플래너가 샘플링 기반이라 결과가 매번 다르다. 기울기가 작게 나오면 이 셀을 다시 실행하면 된다.

In [ ]:
node.get_logger().info('--- A 로 제약 없이 이동 (비교) ---')
plan_viz_execute(
    make_pose(*targets[0]['pos'], math.pi, 0.0, 0.0),
    tol_z=TOL_Z,               # goal yaw 자유도를 4-3 과 동일하게 (공정 비교)
    vel=0.2, label='A',        # path_quat 없음 → ns 는 free_A
)
time.sleep(1.0)

plan_viz_execute(start_pose, label='Back', show=False)
time.sleep(0.5)

### 4-7. 기울기 비교 그래프

RViz 로 안 보여도 이건 반박이 안 된다. 4-3(제약 O)과 4-6(제약 X)의 기울기 곡선을 겹쳐 그리고
허용치 선을 얹는다.

> **백엔드 주의** — 이 환경은 venv 의 `matplotlib_inline` 과 시스템 `matplotlib` 버전이 어긋나
> Jupyter 기본 inline 백엔드가 깨진다 (`RcParams' object has no attribute '_get'`).
> 그래서 **`Agg` 백엔드로 직접 그려 PNG 로 저장한 뒤 이미지를 띄운다.**
> 근본 해결은 둘 중 하나 —
> `pip install -U matplotlib` 또는 `pip install "matplotlib_inline<0.2"` (venv 안에서).

In [ ]:
# ── 1) 숫자 요약 — matplotlib 없이도 항상 동작 ──────────────────────
SERIES = [
    ('con_A',  'with path constraint',    'tab:green', '-'),
    ('free_A', 'without path constraint', 'tab:red',   '--'),
]

print(f"{'':26s}  {'max':>7s} {'mean':>7s}   0°" + ' ' * 26 + '90°')
have = False
for ns, lab, _, _ in SERIES:
    t = TILT_LOG.get(ns)
    if not t:
        print(f'{lab:26s}  (기록 없음 — 해당 셀을 먼저 실행)')
        continue
    have = True
    bar = int(min(max(t) / 90.0, 1.0) * 30)
    print(f'{lab:26s}  {max(t):6.1f}° {sum(t) / len(t):6.1f}°   '
          f'|{"█" * bar}{"·" * (30 - bar)}|')
print(f"{'허용치':26s}  {math.degrees(TOL):6.1f}°")

# ── 2) 그래프 — inline 백엔드를 피해 Agg 로 그린 뒤 PNG 를 띄운다 ────
if have:
    try:
        import matplotlib
        matplotlib.use('Agg', force=True)      # ← matplotlib_inline 을 건드리지 않는다
        import matplotlib.pyplot as plt
        from IPython.display import Image, display

        OUT = 'ex08_tilt_compare.png'
        fig, ax = plt.subplots(figsize=(9, 4), dpi=130)
        for ns, lab, c, ls in SERIES:
            t = TILT_LOG.get(ns)
            if not t:
                continue
            x = [i / (len(t) - 1) * 100 for i in range(len(t))]
            ax.plot(x, t, color=c, ls=ls, lw=2.5,
                    label=f'{lab}  (max {max(t):.1f} deg)')
        ax.axhline(math.degrees(TOL), color='gray', ls=':', lw=1.8,
                   label=f'tolerance +/- {math.degrees(TOL):.0f} deg')
        ax.set_xlabel('trajectory progress [%]')
        ax.set_ylabel('tilt from vertical [deg]')
        ax.set_title('FR3 gripper tilt along the path - target A')
        ax.set_xlim(0, 100)
        ax.set_ylim(bottom=0)
        ax.grid(alpha=0.3)
        ax.legend()
        fig.savefig(OUT, bbox_inches='tight')
        plt.close(fig)
        display(Image(filename=OUT))
        print(f'그래프 저장: {OUT}')
    except Exception as e:
        print(f'그래프 생략 ({type(e).__name__}: {e}) — 위 숫자 요약으로 확인')

### 4-8. ready 복귀

In [21]:
node.get_logger().info('--- ready 복귀 ---')
plan_viz_execute_joint(ready_target, vel=0.3, label='Ready')
node.get_logger().info('=== franka_ex08 완료! ===')

[INFO] [1778402922.844619831] [franka_ex08_constraints_demo]: --- ready 복귀 ---
[INFO] [1778402924.672642819] [franka_ex08_constraints_demo]: === franka_ex08 완료! ===


True

## 5. 정리

In [ ]:
node.destroy_node()
try:
    rclpy.shutdown()
except Exception:
    pass